# Appendix D4: Handling voltage offsets
**Appendix D discusses remaining noise and errors**

Following on from the Appendix on the liquid junction potential (LJP), this notebook discusses how to deal with the LJP and other offsets when fitting models or otherwise analysing experimental data.

Because everybody gets confused about the sign conventions, we'll go through this excruciatingly slowly.

We discuss two situations:

- **Case A**: No previous LJP correction
- **Case B**: "A priori" corrected data

## Case A: No previous LJP correction

In this section, we look at **voltage clamp without prior or posterior LJP correction**.

We shall assume there is a _command potential_ $V_c$, specified in a digital protocol, that gets sent to the amplifier and applied (to the best of its abilities) to the cell.
Neither the digital protocol nor the amplifier settings are adjusted to compensate for the LJP.

We shall also assume that the data has been leak-corrected and/or subtracted.

Four questions:

1. Given the presence of an LJP, what should the _observed_ reversal potential be?
2. How should we quantify and explain a difference between observed and calculated reversal potential?
3. How can we estimate the driving force during the experiment?
4. How do we _simulate_ these offsets with the voltage clamp model?


### 1. Expected observed reversal potential

Let's assume we've calculated a Nernst potential $E_\text{nernst}$, but measured a reversal potential $E_\text{observed}$.
By "measured", we mean "we asked the amplifier to apply $V_c$ volts, and then observed reversal at $V_c = E_\text{observed}$".
What should we expect to observe?

If the LJP is the only distortion, we can find the expected observed reversal potential, $E_\text{expected}$, from the common convention (see LJP appendix):

\begin{align}
V_m &= V_c - V_{LJ} \qquad \text{(Eq. 1)}
\end{align}

Filling in $V_m = E_\text{nernst}$ and $V_c = E_\text{expected}$:

\begin{align}
E_\text{nernst} = E_\text{expected} - V_{LJ} \quad \rightarrow \quad
E_\text{expected} = E_\text{nernst} + V_{LJ}
\end{align}

### 2. Remaining offset

We can quantify the agreement between $E_\text{expected}$ and $E_\text{observed}$ as a "battery", $V_\text{unexplained}$, in the same direction as $E_\text{off}$ in the voltage clamp model (this will be handy later):

\begin{align}
V_m &= V_c + V_\text{unexplained} - V_{LJ} \qquad \text{(Eq. 2)}
\end{align}

As before, we substitute $V_m=E_\text{nernst}$, but now $V_c=E_\text{observed}$:

\begin{align}
E_\text{nernst} = E_\text{observed} + V_\text{unexplained} - V_{LJ}
\quad \rightarrow \quad
V_\text{unexplained} &= E_\text{nernst} + V_{LJ} - E_\text{observed} \\
                     &= E_\text{expected} - E_\text{observed}
\end{align}

This quantity can be useful for quality control.

### 3. Estimating driving force

Assuming a current of the form $I = g \cdot O \cdot (V_m - E)$ it can be useful to estimate $(V_m - E)$, for example to obtain a lower bound on $g$.

#### 3.1 Nernst prediction is correct, but $V_\text{unexplained}$ is present

Assuming that the Nernst prediction is correct, but that a $V_\text{unexplained}$ is present, the driving force is

\begin{align}
V_m - E_\text{nernst} &= V_c - V_{LJ} + V_\text{unexplained} - E_\text{nernst} \\
                      &= V_c - V_{LJ} + E_\text{nernst} + V_{LJ} - E_\text{observed} - E_\text{nernst} \\
                      &= V_c - E_\text{observed} \\
\end{align}

This can feel a bit counterintuitive, since it is tempting to think of $V_c$ as a "clean" input, while $E_\text{observed}$ is a "messy" output. However, we can remind ourselves that neither are "what the cell sees": For the cell $V_m$ and $E_\text{nernst}$ are the true values, while $V_c$ and $E_\text{observed}$ are the values "seen" by the electronics.

We can make this mathematically apparent by forcing the error terms back in again:

\begin{align}
V_m - E_\text{nernst} = (V_c + V_\text{unexplained} - V_{LJ}) - (E_\text{observed} + V_\text{unexplained} - V_{LJ})
\end{align}

which is just the original equation: since the corrections are in both terms, they cancel out.

#### 3.2 Nernst prediction is incorrect, no offset is present

Alternatively, we can decide to trust the experimentally determined reversal over our Nernst-based prediction, as long as we correct for the LJP - leading to a new value $E_\text{experimental}$.

Starting from Equation 1 again
\begin{align}
V_m = V_c - V_{LJ}
\end{align}

this time we fill in $V_m = E_\text{experimental}$ and $V_c = E_\text{observed}$ for

\begin{align}
E_\text{experimental} = E_\text{observed} - V_{LJ}
\end{align}

This gives us a driving force

\begin{align}
V_m - E_\text{experimental} &= (V_c - V_{LJ}) - (E_\text{observed} - V_{LJ}) \\
                            &= V_c - E_\text{observed}
\end{align}

So this is the same result we got by assuming the Nernst prediction was correct, and we would have observed it except for the pesky offset.
And the reason it cancels out is because we determined the offset by assuming the Nernst prediction was right.

#### 3.3 Nernst is correct, but the measured reversal potential is wrong.

Finally, we can decide that our estimation $E_\text{observed}$ was poor, and if we'd done better we would have measured $E_\text{nernst}$.

In this case the driving force is
\begin{align}
V_m - E_\text{nernst} &= V_c - V_{LJ} - E_\text{nernst}
\end{align}

#### 3.4 Conlusion

In summary, if we believe the measured reversal potential, we can explain the difference from $E_\text{nernst}$ as a voltage offset in the experiment (3.1) or an inaccuracy in our prediction (3.2), to find

\begin{align*}
D_\text{reversal experiment is right} = V_c - E_\text{observed}
\end{align*}

If we don't believe the measured reversal potential (3.3), we find

\begin{align*}
D_\text{reversal experiment is wrong} = V_c - V_\text{LJ} - E_\text{nernst}
\end{align*}

The difference between these equations is, by definition, $V_\text{unexplained}$.

### 4. Simulating uncorrected LJP and remaining offset

In the artefact model, we have a quantity $E_\text{off}$ that can be used to represent both the LJP and the unexplained offset.

Ignoring all other distortions:
\begin{align}
V_m = V_c + E_\text{off}
\end{align}

Just like with driving force, we have two options

#### 4.1 Believe the observed reversal potential

In this case, we equate our model $V_m$ with Equation 2 to find
\begin{align}
V_c + E_\text{off} &= V_c + V_\text{unexplained} - V_{LJ} \\
E_\text{off} &= V_\text{unexplained} - V_{LJ} \\
             &= E_\text{nernst} - E_\text{observed}
\end{align}

So with this assumption, (and ignoring series resistance etc.), we just make the simulation reverse at the same $V_c$ as the data by adding an offset $E_\text{nernst} - E_\text{observed}$, and we don't actually need to know the LJP.

#### 4.2 Dismiss the observed reversal potential

In this case, we equate our model $V_m$ with Equation 1 to find
\begin{align}
V_c + E_\text{off} &= V_c - V_{LJ} \\
E_\text{off} = -V_{LJ}
\end{align}

So with this assumption, (and ignoring series resistance etc.), we make the simulation reverse where we think it should reverse, regardless of what the data says.

## Case B: A priori LJP correction

Some other day.